# Languages per Country — World Map

This notebook counts how many languages are spoken in each country according to
the **LinguaMeta** data bundled in `low`, then draws a choropleth world map.

**Requirements**
```
pip install languages-of-the-world plotly pandas
```

In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)

LanguagesOfTheWorld(languages=7929, countries=247, continents=5)


## 2 · Count languages per country

Each `Country` object has a back-reference `.languages` — the list of languages
that list this country as one of their locales (sourced from LinguaMeta).

In [ ]:
rows = [
    {
        "iso2":          country.code,
        "country":       country.label,
        "region":        country.region.label,
        "continent":     country.continent.label,
        "language_count": len(country.languages),
    }
    for country in db.countries
]

df = pd.DataFrame(rows).sort_values("language_count", ascending=False)
df.head(10)

,iso2,country,region,continent,language_count
174,PG,Papua New Guinea,Melanesia,Oceania,858
99,ID,Indonesia,South-eastern Asia,Asia,716
162,NG,Nigeria,Sub-Saharan Africa,Africa,524
103,IN,India,Southern Asia,Asia,426
11,AU,Australia,Australia and New Zealand,Oceania,387
46,CN,China,Eastern Asia,Asia,293
155,MX,Mexico,Latin America and the Caribbean,Americas,287
45,CM,Cameroon,Sub-Saharan Africa,Africa,276
230,US,United States of America,Northern America,Americas,246
29,BR,Brazil,Latin America and the Caribbean,Americas,210


## 3 · Map ISO 3166-1 alpha-2 → alpha-3

Plotly's built-in world geometry uses ISO-3 codes.
We derive the mapping from the same UN M49 CSV that `low.bootstrap` uses.

In [ ]:
import urllib.request, csv, io

_UN_M49_CSV = (
    "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes"
    "/master/all/all.csv"
)

with urllib.request.urlopen(_UN_M49_CSV, timeout=20) as r:
    un_text = r.read().decode("utf-8")

alpha2_to_alpha3 = {
    row["alpha-2"]: row["alpha-3"]
    for row in csv.DictReader(io.StringIO(un_text))
    if row["alpha-2"] and row["alpha-3"]
}

df["iso3"] = df["iso2"].map(alpha2_to_alpha3)
df = df.dropna(subset=["iso3"])
print(f"{len(df)} countries with ISO-3 codes")

247 countries with ISO-3 codes


## 4 · Choropleth map

In [ ]:
fig = px.choropleth(
    df,
    locations="iso3",
    color="language_count",
    hover_name="country",
    hover_data={"iso3": False, "iso2": False, "region": True, "continent": True, "language_count": True},
    color_continuous_scale="YlOrRd",
    range_color=(0, df["language_count"].quantile(0.98)),
    labels={"language_count": "Languages"},
    title="Number of Languages Spoken per Country",
    projection="natural earth",
)

fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    coloraxis_colorbar=dict(
        title="Languages",
        tickfont_size=11,
    ),
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor="#555",
        showland=True,
        landcolor="#f0ede8",
        showocean=True,
        oceancolor="#cce5f0",
        showcountries=True,
        countrycolor="#aaa",
    ),
)

fig.show()

## 5 · Bar chart — top 20 most linguistically diverse countries

In [ ]:
top20 = df.nlargest(20, "language_count")

fig2 = px.bar(
    top20,
    x="language_count",
    y="country",
    orientation="h",
    color="continent",
    text="language_count",
    labels={"language_count": "Number of languages", "country": ""},
    title="Top 20 Countries by Number of Languages Spoken",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=550,
    margin=dict(l=10, r=10, t=40, b=10),
    legend_title_text="Continent",
)
fig2.update_traces(textposition="outside")
fig2.show()

## 6 · Summary statistics

In [ ]:
print("Countries with at least one language recorded:", (df.language_count > 0).sum())
print("Countries with no languages mapped yet:        ", (df.language_count == 0).sum())
print()
print(df.groupby("continent")["language_count"].agg(["sum", "mean", "max"]).round(1))

Countries with at least one language recorded: 247
Countries with no languages mapped yet:         0

            sum  mean  max
continent                 
Africa     2604  43.4  524
Americas   1312  23.0  287
Asia       2592  51.8  716
Europe      371   7.3  113
Oceania    1548  53.4  858
